# 🧠 LOGOS — Vedic-Physics Hybrid LLM Training on Kaggle
**C++20 | Vedic GEMM | Langevin Dynamics Optimizer**

## STEP 1 — Environment Check

In [ ]:
import os, glob, subprocess, sys, re, math, shutil
print('=== GPU ===')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print('\n=== Compiler ===')
os.system('g++ --version | head -1')
print('\n=== CMake ===')
os.system('cmake --version | head -1')
print('\n=== CPU Cores ===')
os.system('nproc')
print('\n=== Disk ===')
os.system('df -h /kaggle/working')

## STEP 2 — Dataset Setup
`.bin` file ko decode karke `.txt` mein convert karenge

In [ ]:
# Saare input files dekhte hain
print('=== All input files ===')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full)
        print(f'  {full}  ({size//1024} KB)')

In [ ]:
# train.bin ko text mein convert karo
# GPT-2 tokenizer se encoded hai — tiktoken se decode karenge

BIN_FILE = '/kaggle/input/datasets/josephmayok/roneneldan-tinystories/train.bin'
TRAIN_TXT = '/kaggle/working/dataset.txt'

# tiktoken install karo (Kaggle pe available)
os.system('pip install tiktoken -q')

import tiktoken
import numpy as np

print('Decoding train.bin → dataset.txt ...')

# GPT-2 tokenizer
enc = tiktoken.get_encoding('gpt2')

# Binary file read karo (uint16 tokens)
data = np.fromfile(BIN_FILE, dtype=np.uint16)
print(f'Total tokens in bin: {len(data):,}')

# Decode karo — chunks mein (memory safe)
CHUNK = 100000  # 100K tokens at a time
total_written = 0

with open(TRAIN_TXT, 'w', encoding='utf-8') as f:
    for i in range(0, len(data), CHUNK):
        chunk = data[i:i+CHUNK].tolist()
        text = enc.decode(chunk)
        f.write(text)
        total_written += len(chunk)
        if i % (CHUNK * 10) == 0:
            print(f'  Decoded {total_written:,} tokens...')

size_mb = os.path.getsize(TRAIN_TXT) / 1024 / 1024
print(f'\n✅ dataset.txt ready: {size_mb:.1f} MB')
print('First 300 chars:')
with open(TRAIN_TXT) as f:
    print(f.read(300))

## STEP 3 — Clone & Build LOGOS

In [ ]:
WORK_DIR = '/kaggle/working/LOGOS'

if not os.path.exists(WORK_DIR):
    print('Cloning LOGOS...')
    os.system(f'git clone https://github.com/Vikas8719/LOGOS.git {WORK_DIR}')
else:
    print('Already cloned — pulling latest...')
    os.system(f'cd {WORK_DIR} && git pull')

os.chdir(WORK_DIR)
print('\nFiles:')
os.system('ls include/ src/')

In [ ]:
os.chdir(WORK_DIR)
print('Building LOGOS with -O3 -march=native ...')

ret = os.system('''
    cmake -B build \
        -DCMAKE_BUILD_TYPE=Release \
        -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
    && cmake --build build --parallel $(nproc)
''')

if ret == 0:
    print('\n✅ Build SUCCESS')
    os.system('ls -lh build/logos')
else:
    print('\n❌ Build FAILED')

## STEP 4 — Tests

In [ ]:
os.chdir(WORK_DIR)
os.system('./build/logos --test')
os.system('./build/logos --forward')
os.system('./build/logos --benchmark')

## STEP 5 — 🚀 TRAIN LOGOS
```
Step     0 → Loss ~8.3  (random)
Step   500 → Loss ~5.0  (patterns)
Step  5000 → Loss ~3.0  (meaningful)
```

In [ ]:
os.chdir(WORK_DIR)
TRAIN_FILE = '/kaggle/working/dataset.txt'
LOG_FILE   = '/kaggle/working/training_log.txt'

print('🚀 LOGOS Training START')
print(f'   Dataset: {os.path.getsize(TRAIN_FILE)//1024//1024} MB')
print('   Ctrl+C se rok sakte ho — checkpoint auto-save hoga\n')

steps_log, losses_log = [], []

process = subprocess.Popen(
    ['./build/logos', '--train', TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                steps_log.append(int(m.group(1)))
                losses_log.append(float(m.group(2)))
    except KeyboardInterrupt:
        process.terminate()
        print('\n⏹️  Stopped by user')

process.wait()
if losses_log:
    print(f'\nStart loss : {losses_log[0]:.4f}')
    print(f'Final loss : {losses_log[-1]:.4f}')
    print(f'Improvement: {losses_log[0]-losses_log[-1]:.4f} ✅')

## STEP 6 — Loss Curve

In [ ]:
import matplotlib.pyplot as plt

LOG_FILE = '/kaggle/working/training_log.txt'
steps_log, losses_log = [], []

with open(LOG_FILE) as f:
    for line in f:
        m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
        if m:
            steps_log.append(int(m.group(1)))
            losses_log.append(float(m.group(2)))

if steps_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps_log, losses_log, 'b-', linewidth=1.5)
    ax1.set_title('LOGOS Training Loss (Langevin Optimizer)')
    ax1.set_xlabel('Steps'); ax1.set_ylabel('Loss')
    ax1.axhline(y=losses_log[-1], color='r', linestyle='--',
                label=f'Final: {losses_log[-1]:.3f}')
    ax1.grid(True, alpha=0.3); ax1.legend()

    perp = [math.exp(min(l, 10)) for l in losses_log]
    ax2.plot(steps_log, perp, 'g-', linewidth=1.5)
    ax2.set_title('Perplexity (Lower = Better)')
    ax2.set_xlabel('Steps'); ax2.set_ylabel('Perplexity')
    ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150)
    plt.show()

    print(f'Start  → Loss: {losses_log[0]:.4f} | PPL: {math.exp(losses_log[0]):.1f}')
    print(f'Final  → Loss: {losses_log[-1]:.4f} | PPL: {math.exp(min(losses_log[-1],10)):.1f}')

## STEP 7 — Evaluate + Generate

In [ ]:
os.chdir(WORK_DIR)
TRAIN_FILE = '/kaggle/working/dataset.txt'

ckpts = sorted([f for f in glob.glob('*.bin') if 'vocab' not in f])
print('Checkpoints:', ckpts)

if ckpts:
    latest = ckpts[-1]
    print(f'\n📊 Eval: {latest}')
    os.system(f'./build/logos --eval {TRAIN_FILE} {latest}')

    print('\n=== TEXT GENERATION ===')
    for p in ['Once upon a time', 'The little girl', 'Tom liked to']:
        print(f"\nPrompt: '{p}'")
        os.system(f'./build/logos --generate {latest} "{p}"')
else:
    print('⚠️  Pehle Step 5 (training) chalao')

## STEP 8 — Save Output

In [ ]:
os.chdir(WORK_DIR)
OUT = '/kaggle/working/logos_trained'
os.makedirs(OUT, exist_ok=True)

for f in glob.glob('*.bin'):  shutil.copy(f, OUT); print(f'✅ {f}')
shutil.copy('build/logos', OUT);  print('✅ logos binary')
for f in ['/kaggle/working/loss_curve.png', '/kaggle/working/training_log.txt']:
    if os.path.exists(f): shutil.copy(f, OUT); print(f'✅ {os.path.basename(f)}')

print(f'\n=== Output ===')
os.system(f'ls -lh {OUT}')
print('\n✅ Kaggle Output tab se download kar sakte ho!')